# Project interactive visualization
Using a dataset that your group is consider using for the term project, let's build some meaningful user-driven data visualization. Depending on your dataset this could include:
- Usage of geospatial information
- Building interactive views with widgets
- Organize multiple components into a simple dashboard

Keep these design principles in mind:
While you navigate through this notebook some things to take into consideration:
* Do not add interaction just to add it. Make sure it helps answer a question.
* Use meaningful titles and labels
* Document your code so it's readable and clean. If something does not work, document the issue and explain your best attempt.

In [ ]:
## These are most likely the libraries you will use
# Add or remove imports as needed

# Install required packages
!pip install -q plotly geopy panel jupyter_bokeh

import pandas as pd
import numpy as np

# Visualization libraries
import plotly.express as px
import plotly.graph_objects as go

# Geospatial / geocoding
from geopy.geocoders import Nominatim

# Panel
import panel as pn
pn.extension('plotly')

In [ ]:
### Load your dataset here

# Example:
# df = pd.read_csv("your_dataset.csv")
from datasets import load_dataset
import pandas as pd
dataset = load_dataset("electricsheepafrica/nigerian-banking-bnpl")
df = dataset["train"].to_pandas()

In [ ]:
df

In [ ]:
# Write your code here
df.info()

## Question 1: Analytical Question
Write a question about your data that can be explored with an interactive plot. A good example would be a question where you can compare different categories. Explain how the plot help answer your question and why you choose that plot type.

Examples:
- Which regions have the highest average values?
- How does one variable compare across time?

### Question:  
How does the 90-day default rate change based on credit score and provider?

## Question 2. Create a simple interaction plot
Create a plot, it can be related to your question #1 or a new question, that users can interact with in some meaningful way. Explain in a markdown, how does the interaction add to the analysis.

Example of possible interactions:
*   Hover over information
*   Toogle between groups

In [ ]:
# Your code . . .
default_rate = df.groupby(["credit_score", "provider"])["default_90d"].mean().reset_index()

q2 = px.line(default_rate, x="credit_score", y="default_90d", color="provider",
              title="90-day default rate by credit score and provider",
              labels={"credit_score": "credit score", "default_90d": "default rate"})
q2.show()

At any given point, you can see both the provider, credit score, and default rate when hovering over it. You can also filter by each provider individually which could help someone to choose their provider based on their credit score.

## Question 3. Choropleth Planning
Design a choropleth idea using your dataset.
In a markdown cell:
*  Identify the geographic unit you would map (state, county, country, ZIP code, etc.)
*  Identify the variable you would color by
*  Explain if any aggregation or preprocessing is needed
*  Briefly describe what GeoJSON file would be required

You do not need to have a perfect dataset for this question. However, your plan should be realistic.

If your data does not fully support a choropleth, build a prototype table that explains that structure you would need before mapping.

**Geographic unit:**  
I would map the data at the **state level in Nigeria** using the `customer_state` column.

**Variable to color by:**  
A good variable to color by would be the **average 90-day default rate (`default_90d`)** for each state. This would let us compare which states appear to have higher or lower BNPL repayment risk.

**Aggregation / preprocessing needed:**  
Yes. Since each row is an individual BNPL transaction, the data should first be grouped by `customer_state`. After grouping, I would calculate a summary statistic such as:
- average `default_90d` rate by state, or
- average `credit_score` by state, or
- total BNPL purchase amount (`principal_ngn`) by state.

I would also clean state names so they match the map file exactly. For example, `Abuja (FCT)` may need to be renamed to `Federal Capital Territory` depending on the GeoJSON labels.

**GeoJSON file required:**  
I would need a **GeoJSON file of Nigeria's state boundaries** where each feature represents one Nigerian state. The GeoJSON should include a state name field that matches the cleaned values in `customer_state` so the dataset can be joined correctly.

This choropleth would be useful for seeing whether BNPL risk or usage patterns vary across different parts of Nigeria.

## Question 4. Geospatial Possibility Check

Determine whether your dataset can support a map-based visualization.

In a markdown cell, answer one of the following:
- If **yes**, explain what geographic fields you have and what type of map is appropriate.
- If **no**, explain what is missing and what you would need to create a map.

Write code that inspects or prepares the geographic column(s) you may use.

**Yes, this dataset can support a map-based visualization.**  
The main geographic field available is `customer_state`, which gives the Nigerian state or territory associated with each BNPL record. Because this is a categorical location field rather than latitude/longitude coordinates, the most appropriate map would be a **state-level choropleth map of Nigeria**.

A choropleth would work well because we can aggregate the dataset by state and then shade each state based on a summary value such as:
- average 90-day default rate,
- average credit score,
- total transaction amount, or
- number of BNPL transactions.

What the dataset does **not** appear to include is exact coordinates, addresses, or city-level point locations, so a point map is not the best choice right now. To create the choropleth, we would need a GeoJSON file for Nigerian states and make sure the state names in the dataset match the names in that file. One likely cleaning step is converting `Abuja (FCT)` into the exact label used in the GeoJSON, such as `Federal Capital Territory`.

In [ ]:
# Your code . . .

# Inspect geographic column(s) that could support a map
geo_cols = [col for col in df.columns if "state" in col.lower() or "city" in col.lower() or "country" in col.lower() or "location" in col.lower()]
print("Possible geographic columns:", geo_cols)

# Look at the state values we have
state_counts = df["customer_state"].value_counts().sort_index()
print("\nNumber of unique states/territories:", df["customer_state"].nunique())
print("\nSample state values:")
print(state_counts.head(15))

# Prepare a state-level summary that could later be joined to a Nigeria states GeoJSON
state_map_data = (
    df.groupby("customer_state", as_index=False)
      .agg(
          avg_default_90d=("default_90d", "mean"),
          avg_credit_score=("credit_score", "mean"),
          total_principal_ngn=("principal_ngn", "sum"),
          transaction_count=("customer_id", "count")
      )
      .sort_values("avg_default_90d", ascending=False)
)

# Clean names if needed for a future map join
state_map_data["state_clean"] = state_map_data["customer_state"].replace({
    "Abuja (FCT)": "Federal Capital Territory"
})

state_map_data.head()

## Question 5. Geopy / Location Preparation

If your dataset has location names, addresses, cities, states, or countries, use geopy on a sample of your data to geocode locations or validate location information.

If your dataset does not have data that contains location, pick 5 destination you want to visit and use geopy to geocode locations.

In [ ]:
# Your code . . .

from geopy.geocoders import Nominatim
import time

# Initialize geocoder
geolocator = Nominatim(user_agent="bnpl_project")

# Take a sample of unique states (keep it small so it runs fast)
sample_states = df["customer_state"].dropna().unique()[:5]

# Store results
geo_results = []

for state in sample_states:
    try:
        location = geolocator.geocode(f"{state}, Nigeria")
        if location:
            geo_results.append({
                "state": state,
                "latitude": location.latitude,
                "longitude": location.longitude
            })
        else:
            geo_results.append({
                "state": state,
                "latitude": None,
                "longitude": None
            })
        time.sleep(1)  # avoid rate limiting
    except Exception as e:
        geo_results.append({
            "state": state,
            "latitude": None,
            "longitude": None
        })

# Convert to DataFrame
geo_df = pd.DataFrame(geo_results)

geo_df

I used geopy to convert a sample of the customer_state values into latitude and longitude coordinates. Since the dataset already includes state names, I passed each state along with “Nigeria” into the geocoder to get approximate locations.

I only used a small sample of states to avoid hitting rate limits from the API. The results give a central point for each state, which could be useful for plotting points on a map or checking location accuracy.

This shows that the dataset can be extended with geographic coordinates, although for this project a choropleth map is still more appropriate than a point-based map.

## Question 6. Panel Widget

Create a Panel Widget that controls something in your analysis such as the ability to choose a column, category, year, etc.

The widget should affect an output such as a plot, table, or summary statistic.

In [ ]:
# Your code . . .

import panel as pn
pn.extension('plotly')

# Create dropdown widget
provider_select = pn.widgets.Select(
    name="Select Provider",
    options=["All"] + sorted(df["provider"].dropna().unique().tolist()),
    value="All"
)

# Function to update plot
def update_plot(provider):
    if provider == "All":
        filtered = df
    else:
        filtered = df[df["provider"] == provider]

    temp = (
        filtered
        .groupby("credit_score")["default_90d"]
        .mean()
        .reset_index()
    )

    fig = px.line(
        temp,
        x="credit_score",
        y="default_90d",
        title=f"Default Rate vs Credit Score ({provider})"
    )

    return fig

# Bind widget to function
interactive_plot = pn.bind(update_plot, provider_select)

# Layout
dashboard = pn.Column(provider_select, interactive_plot)

dashboard

I created a Panel dropdown widget that allows the user to select a provider. Based on the selected provider, the plot updates to show the relationship between credit score and average 90-day default rate.

If “All” is selected, the plot includes data from all providers. Otherwise, the dataset is filtered to only include the selected provider before calculating the averages.

This makes the analysis more interactive and helps compare how default risk varies across different providers in a more focused way.

## Question 7. Mini Dashboard

Build a small Panel dashboard with at least two components. Arrange the components so that it is in a readable layout. Your dashboard should include:
* At least one plot,
* An additional element of your choice such as a widget, table, second plot, etc.

In [ ]:
import panel as pn
import plotly.express as px

pn.extension('plotly')

#create the widegets for the three types
default_selector = pn.widgets.Select(
    name='Default Window',
    options=['default_30d', 'default_90d'],
    value='default_30d'
)

customer_type_selector = pn.widgets.Select(
    name='Customer Type',
    options=['All', 'First-Time', 'Returning'],
    value='All'
)

state_selector = pn.widgets.Select(
    name='Customer State',
    options=['All'] + sorted(df['customer_state'].unique().tolist()),
    value='All'
)

#helper function for quick filtration
def get_default_stats(df, default_col, customer_type, state):
    filtered = df

    if customer_type == 'First-Time':
        filtered = filtered[filtered['first_time_customer'] == True]
    elif customer_type == 'Returning':
        filtered = filtered[filtered['first_time_customer'] == False]

    if state != 'All':
        filtered = filtered[filtered['customer_state'] == state]

    return filtered.groupby('merchant_category').agg(
        default_rate=(default_col, 'mean'),
        avg_principal=('principal_ngn', 'mean'),
        count=('transaction_id', 'count')
    ).reset_index().sort_values('default_rate', ascending=False).round(3)

#used to update the dashboard
def update_dashboard(default_col, customer_type, state):
    stats = get_default_stats(df, default_col, customer_type, state)

    fig = px.bar(
        stats, x='merchant_category', y='default_rate',
        title=f'Default Rate by Merchant Category ({default_col}) | {customer_type} Customers | State: {state}',
        labels={'default_rate': 'Default Rate', 'merchant_category': 'Category'},
        color='default_rate', color_continuous_scale='Reds'
    )

    return pn.Row(
        pn.pane.Plotly(fig),
        pn.pane.DataFrame(stats, index=False)
    )

bound_dashboard = pn.bind(update_dashboard, default_selector, customer_type_selector, state_selector)

dashboard = pn.Column(
    "## BNPL Default Risk Dashboard",
    pn.Row(default_selector, customer_type_selector, state_selector),
    bound_dashboard
)

dashboard

In [ ]:
import panel as pn
import plotly.express as px
import pandas as pd

pn.extension('plotly')

# widget for the slider
credit_score_slider = pn.widgets.IntRangeSlider(
    name='Credit Score Range',
    start=int(df['credit_score'].min()),
    end=int(df['credit_score'].max()),
    value=(int(df['credit_score'].min()), int(df['credit_score'].max())),
    step=10
)

@pn.depends(credit_score_slider.param.value)
def update_dashboard(score_range):
    low, high = score_range
    filtered = df[(df['credit_score'] >= low) & (df['credit_score'] <= high)]

    #stats fro the different categories
    stats = filtered.groupby('merchant_category').agg(
        default_rate_30d=('default_30d', 'mean'),
        default_rate_90d=('default_90d', 'mean'),
        avg_principal=('principal_ngn', 'mean'),
        count=('transaction_id', 'count')
    ).reset_index().sort_values('default_rate_30d', ascending=False).round(3)

    #create the bar chart
    fig = px.bar(
        stats, x='merchant_category', y=['default_rate_30d', 'default_rate_90d'],
        title=f'Default Rate by Merchant Category | Credit Score: {low}–{high}',
        labels={'value': 'Default Rate', 'merchant_category': 'Category', 'variable': 'Window'},
        barmode='group',
        color_discrete_map={
            'default_rate_30d': '#e07b54',
            'default_rate_90d': '#c0392b'
        }
    )

    return pn.Column(
        pn.Row(
            pn.pane.Plotly(fig, sizing_mode='stretch_width'),
        ),
        pn.pane.DataFrame(stats, index=False)
    )

dashboard = pn.Column(
    "## BNPL Risk Profile Explorer",
    credit_score_slider,
    update_dashboard
)

dashboard.servable()

## Question 8. Reflection

Write a short reflection addressing all of the following:
- Which interactive element was most useful in your notebook?
- What was the hardest part of working with your dataset?
- Did implementing interactive tools help your analysis? Why or why not?
- If you had more time, what would you improve or add?

* We found our '90 day default rate by credit score and provider' chart to be both interesting and useful. This chart helps us answer the question of how a 90 day default rate can vary by credit score and also by provider. When deciding to use a Buy Now Pay Later payment method, choosing your provider is your first step. This chart lets you interactively compare and contrast the different providers, along with the default rate at your particular credit score.

* The most difficult part of working with this dataset has been the size of it. Considering it has two-million rows, many of the data trends apperared to be very similar. With this said, many of the plots we created gave us limited information, and we were given no option but to create more specific plots to help answer our questions

* Implementing interacitve tools certainly helped our analysis, particulary when it came to comparing, contrasting, and filtering. As mentioned in the previous respone, one of our interactive graphs were used to compare 90 day defualt rates by credit score and provider. Another interactive tool allowed us to view quick insights of the data by filtering through default window, customer type, and customer state. Another graph does something similar, allowing us to slide a bar representing credit score range to view relevant insights. Using these interactive tools, we were able to answer multiple questions for each interactive tool produced.

* If we had more time, something neat and useufl we can add is a choropleth map, where you can view default rates (perhaps other categories too) by customer states. A choropleth map could give us information we may have not addressed, and can give us further important insights. Additioanlly, we never made use of the purchase_date and first_payment_due columns. It is certianly possible that these two columsn could give us useful insights. Plotting default rates over time using a timeline could expose seasonal patterns and much more.